# PicoCal - Cell window vs containment: attacking the real floor (notebook 18)

nb16 error analysis: the min-bias ~0.056 floor is **containment fluctuation** - |residual| correlates **0.977** with |1 - captured/true|. kNN-25 (tuned on clean signal) misses energy that leaks outside the window (under-containment) and admits pileup inside it (over-containment). This notebook varies the **cell window (kNN-25 / 40 / 60)** with the **in-time gated** spacetime model:
- more cells -> recover leaked signal (fix under-containment),
- in-time gate -> reject the extra pileup the bigger window drags in (fix over-containment).

For each window we report the **containment spread** (the driver) and the gated-model σ_eff. Target 0.04: if a larger window + gate tightens containment and σ_eff drops toward 0.04, the floor was selection, not the model.

In [1]:
import sys, copy, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import split, resolution, EPS

WINDOWS = [25, 40, 60]
SEEDS = 3
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 256, "epochs": 150, "patience": 25, "pair_hidden": 32}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
in_dim, n_global = 15, 5
{"windows": WINDOWS, "device": DEVICE}

{'windows': [25, 40, 60], 'device': 'cuda'}

In [2]:
def pair_feats(R):
    rx, ry, le = R[..., 0], R[..., 1], R[..., 2]
    dx = rx.unsqueeze(2) - rx.unsqueeze(1); dy = ry.unsqueeze(2) - ry.unsqueeze(1)
    dR = torch.sqrt(dx * dx + dy * dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1); emin = torch.minimum(le.unsqueeze(2), le.unsqueeze(1))
    return torch.stack([dx, dy, dR, esum, emin], -1)

class PairEmbed(nn.Module):
    def __init__(self, nh, h):
        super().__init__(); self.net = nn.Sequential(nn.Linear(5, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, nh))
    def forward(self, pf): return self.net(pf).permute(0, 3, 1, 2).contiguous()

class PMHA(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.h = nh; self.dh = d // nh
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d); self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        B, L, d = x.shape
        q = self.q(x).view(B, L, self.h, self.dh).transpose(1, 2); k = self.k(x).view(B, L, self.h, self.dh).transpose(1, 2); v = self.v(x).view(B, L, self.h, self.dh).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5) + U
        s = s.masked_fill((~kv).view(B, 1, 1, L), -1e9)
        return self.o((self.drop(s.softmax(-1)) @ v).transpose(1, 2).reshape(B, L, d))

class Block(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nh, drop); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d)); self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        x = x + self.drop(self.attn(self.n1(x), U, kv)); return x + self.drop(self.ff(self.n2(x)))

class PairT(nn.Module):
    def __init__(self):
        super().__init__(); d = cfg["d"]
        self.embed = nn.Linear(in_dim, d); self.pair = PairEmbed(cfg["nhead"], cfg["pair_hidden"])
        self.blocks = nn.ModuleList([Block(d, cfg["nhead"], cfg["dropout"]) for _ in range(cfg["layers"])])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + n_global, d), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(d, 1))
        self.gate = nn.Sequential(nn.Linear(2, 16), nn.GELU(), nn.Linear(16, 1))
    def forward(self, x, m, w, g, base, R):
        U = self.pair(pair_feats(R)); h = self.embed(x)
        for blk in self.blocks: h = blk(h, U, m)
        gate = torch.sigmoid(self.gate(R[..., 3:5]).squeeze(-1)) * m.float()
        pw = w * gate; pw = pw / pw.sum(1, keepdim=True).clamp(min=1e-9)
        p = self.norm((h * pw.unsqueeze(-1)).sum(1))
        return base + self.head(torch.cat([p, g], 1))

In [3]:
def build_tensors(D):
    y = D["y"]; Et = D["Etrue"]; agg = D["agg"]
    keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
    ktr, kva, kte = (keep[s] for s in split(len(keep)))
    G = np.stack([agg[:, 0], agg[:, 3], np.log(agg[:, 2] + 1.0), agg[:, 1], agg[:, 4]], 1).astype(np.float32)
    la, lb = np.polyfit(agg[ktr, 0], y[ktr], 1); base_all = (la * agg[:, 0] + lb).astype(np.float32)
    N = len(y); maxL = max(t.shape[0] for t in D["tok15"])
    X = np.zeros((N, maxL, in_dim), np.float32); M = np.zeros((N, maxL), np.bool_)
    R = np.zeros((N, maxL, 5), np.float32); W = np.zeros((N, maxL), np.float32)
    for i, t in enumerate(D["tok15"]):
        L = t.shape[0]; X[i, :L] = t; M[i, :L] = True; R[i, :L] = D["R"][i]
        e = np.expm1(np.clip(t[:, 0], 0, None)); W[i, :L] = e / (e.sum() + 1e-9)
    cont = X[ktr][:, :, :9].reshape(-1, 9)[M[ktr].reshape(-1)]; mean = cont.mean(0); std = cont.std(0) + EPS
    X[:, :, :9] = (X[:, :, :9] - mean) / std; X[~M] = 0.0
    gm = G[ktr].mean(0); gs = G[ktr].std(0) + EPS; Gn = ((G - gm) / gs).astype(np.float32)
    T = lambda z: torch.from_numpy(z).to(DEVICE)
    tn = dict(X=T(X), M=T(M), R=T(R), W=T(W), G=T(Gn), B=T(base_all).unsqueeze(1), Y=T(y.astype(np.float32)).unsqueeze(1),
              ktr=ktr, kva=kva, kte=kte, y=y, Et=Et, agg=agg, la=la, lb=lb)
    return tn

def train_eval(tn, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = PairT().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            b = torch.from_numpy(idx[j:j + bs]).to(DEVICE)
            yield tn["X"][b], tn["M"][b], tn["W"][b], tn["G"][b], tn["B"][b], tn["R"][b], tn["Y"][b]
    def run(idx):
        out = []
        with torch.no_grad():
            for X, m, w, g, base, R, _ in batches(idx, 512, False):
                out.append(model(X, m, w, g, base, R).cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X, m, w, g, base, R, yb in batches(tn["kva"], 512, False):
                s += nn.functional.mse_loss(model(X, m, w, g, base, R), yb).item(); c += 1
        return s / max(c, 1)
    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X, m, w, g, base, R, yb in batches(tn["ktr"], cfg["batch"], True):
            opt.zero_grad(); nn.functional.mse_loss(model(X, m, w, g, base, R), yb).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]: break
    model.load_state_dict(bstate); model.eval()
    a, b = np.polyfit(run(tn["kva"]), tn["y"][tn["kva"]], 1)
    return float(resolution(np.exp(a * run(tn["kte"]) + b), tn["Et"][tn["kte"]])["sigma_eff"])

In [ ]:
rows = []
for K in WINDOWS:
    with open(repo / "data" / "cache" / f"minbias_spacetime_knn{K}.pkl", "rb") as f:
        D = pickle.load(f)
    tn = build_tensors(D)
    tr, te = tn["ktr"], tn["kte"]; Et = tn["Et"]; agg = tn["agg"]
    sumE = np.expm1(agg[:, 0]); Cn = (sumE / Et); Cn = Cn / np.median(Cn)
    cont_iqr = float((np.percentile(Cn[te], 75) - np.percentile(Cn[te], 25)) / 1.349)
    sum_sig = float(resolution(np.exp(tn["la"] * agg[te, 0] + tn["lb"]), Et[te])["sigma_eff"])
    gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[tr], tn["y"][tr])
    bdt = float(resolution(np.exp(gb.predict(agg[te])), Et[te])["sigma_eff"])
    vals = [train_eval(tn, s) for s in range(SEEDS)]
    med_cells = int(np.median([t.shape[0] for t in D["tok15"]]))
    rows.append({"window": f"kNN-{K}", "med_cells": med_cells, "containment_IQR": round(cont_iqr, 3),
                 "sum": round(sum_sig, 4), "BDT": round(bdt, 4),
                 "gated_transformer": round(float(np.mean(vals)), 4), "std": round(float(np.std(vals)), 4)})
    print(f"kNN-{K}: cont_IQR {cont_iqr:.3f}  sum {sum_sig:.4f}  gated {np.mean(vals):.4f} +/- {np.std(vals):.4f}", flush=True)
summary18 = pd.DataFrame(rows)
summary18

In [ ]:
import plotly.graph_objects as go
d = summary18
fig = go.Figure()
fig.add_trace(go.Bar(name="gated transformer", x=d["window"], y=d["gated_transformer"],
                     error_y=dict(type="data", array=d["std"]), marker_color="#2ca02c"))
fig.add_trace(go.Bar(name="calibrated sum", x=d["window"], y=d["sum"], marker_color="#d62728"))
fig.add_hline(y=0.04, line_dash="dot", line_color="green", annotation_text="target 0.04")
fig.add_hline(y=0.036, line_dash="dot", line_color="gray", annotation_text="clean floor ~0.036")
fig.update_layout(barmode="group", template="plotly_white", height=450, yaxis_title="sigma_eff",
                  title="σ_eff vs cell window (in-time gated model)")
fig.show()
fig2 = go.Figure(go.Scatter(x=d["med_cells"], y=d["containment_IQR"], mode="lines+markers+text",
                            text=d["window"], textposition="top center", line=dict(color="#1f77b4")))
fig2.update_layout(template="plotly_white", height=380, xaxis_title="median cells", yaxis_title="containment spread (IQR/1.349)",
                   title="Does a bigger window tighten containment?")
fig2.show()

## Read-out
- **Does the window fix containment?** If `containment_IQR` drops from kNN-25 → 40 → 60, the bigger window is recovering leaked energy. If σ_eff follows it down toward 0.04, the floor was cell selection.
- **Gate vs pileup:** the gated model should tolerate the bigger window better than the raw sum (which just adds the extra pileup) - watch the gap between `gated_transformer` and `sum` widen at kNN-60.
- If σ_eff flattens despite better containment, the residual is intrinsic (stochastic term / un-subtractable in-time pileup) and 0.04 needs a cleaner sample.